# 03 — Demand Forecasting
## Student 2 (ML Engineer) — Model 1: `next_7_day_demand` Regression

This notebook trains, validates, and compares demand forecasting models.

**Key principles:**
- **Baseline first** — every model must beat the naive 7-day rolling mean
- **Time-aware CV** — `TimeSeriesSplit(n_splits=5)`, never random shuffle
- **Permitted algorithms only** — Linear Regression, Decision Tree, Random Forest, XGBoost
- **MAE is the headline metric** — directly interpretable as 'average units off'
- **Segment-level breakdown** — honest per-ABC/per-store-type metrics

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False

sys.path.insert(0, os.path.join('..', 'src'))

sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (12, 6)

print('Setup complete. XGBoost:', 'Available' if HAS_XGBOOST else 'Not installed')

## 1. Load Feature Table

In [ ]:
FEATURE_TABLE_PATH = os.path.join('..', 'data', 'processed', 'feature_table.csv')
TARGET = 'next_7_day_demand'
HOLDOUT_DAYS = 14

df = pd.read_csv(FEATURE_TABLE_PATH, parse_dates=['date'])
print(f'Feature table: {len(df):,} rows × {len(df.columns)} columns')
print(f'Date range: {df["date"].min().date()} → {df["date"].max().date()}')

# Drop rows without target
df = df.dropna(subset=[TARGET])
print(f'Rows with valid target: {len(df):,}')
df.head()

## 2. Feature Selection & Train/Test Split

In [ ]:
# Leakage-safe features only
FEATURE_COLS = [
    'day_of_week', 'weekend_flag', 'month', 'week_no',
    'lag_1', 'lag_7', 'lag_14',
    'rolling_mean_7', 'rolling_mean_14', 'rolling_std_7',
    'days_of_inventory', 'inventory_to_demand_ratio', 'reorder_gap',
    'discount_pct', 'price_change', 'promotion_flag',
]

# Add optional features if they exist
optional = ['festival_flag', 'holiday', 'temp_c', 'rain_mm', 'local_event',
            'shelf_life_days', 'lead_days', 'floor_area_sqft', 'avg_daily_customers']
features = [c for c in FEATURE_COLS + optional if c in df.columns]

# One-hot encode categoricals
cat_cols = [c for c in ['store_type', 'category', 'brand', 'abc_class', 'xyz_class'] if c in df.columns]
if cat_cols:
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)
    dummy_cols = [c for c in df.columns if any(c.startswith(cat + '_') for cat in cat_cols)]
    features.extend(dummy_cols)

# Sort chronologically & fill NaN
df = df.sort_values('date').reset_index(drop=True)
df[features] = df[features].fillna(0)

# Time-aware holdout split
cutoff = df['date'].max() - pd.Timedelta(days=HOLDOUT_DAYS)
train = df[df['date'] <= cutoff]
test = df[df['date'] > cutoff]

X_train, y_train = train[features], train[TARGET]
X_test, y_test = test[features], test[TARGET]

print(f'Features: {len(features)}')
print(f'Train: {len(train):,} rows (up to {cutoff.date()})')
print(f'Test:  {len(test):,} rows (last {HOLDOUT_DAYS} days)')

## 3. Baseline — 7-Day Rolling Mean
This is the sanity floor. Every real model must beat this.

In [ ]:
baseline_pred = test['rolling_mean_7'].fillna(0) * 7 if 'rolling_mean_7' in test.columns else pd.Series(0, index=test.index)

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_r2 = r2_score(y_test, baseline_pred)

print(f'Baseline (7-day mean × 7):')
print(f'  MAE:  {baseline_mae:.4f}')
print(f'  RMSE: {baseline_rmse:.4f}')
print(f'  R²:   {baseline_r2:.4f}')

## 4. Cross-Validation (TimeSeriesSplit, k=5)
No random shuffling — each fold trains on the past and validates on the next chronological block.

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(max_depth=10, min_samples_leaf=20, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=15, min_samples_leaf=10, n_jobs=-1, random_state=42),
}
if HAS_XGBOOST:
    models['XGBoost'] = XGBRegressor(n_estimators=300, max_depth=8, learning_rate=0.05,
                                     subsample=0.8, colsample_bytree=0.8, n_jobs=-1,
                                     random_state=42, verbosity=0)

tscv = TimeSeriesSplit(n_splits=5)
cv_results = {}

for name, model in models.items():
    fold_maes = []
    for train_idx, val_idx in tscv.split(X_train):
        from sklearn.base import clone
        m = clone(model)
        m.fit(X_train.iloc[train_idx], y_train.iloc[train_idx])
        pred = m.predict(X_train.iloc[val_idx])
        fold_maes.append(mean_absolute_error(y_train.iloc[val_idx], pred))
    cv_results[name] = {'CV MAE (mean)': np.mean(fold_maes), 'CV MAE (std)': np.std(fold_maes)}
    print(f'{name:20s} — CV MAE: {np.mean(fold_maes):.4f} ± {np.std(fold_maes):.4f}')

pd.DataFrame(cv_results).T

## 5. Final Training & Holdout Evaluation

In [ ]:
results = {}
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = np.clip(model.predict(X_test), 0, None)
    results[name] = {
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'R²': r2_score(y_test, pred),
    }
    trained_models[name] = (model, pred)

# Add baseline for comparison
results['Baseline (7-day mean)'] = {'MAE': baseline_mae, 'RMSE': baseline_rmse, 'R²': baseline_r2}

results_df = pd.DataFrame(results).T.sort_values('MAE')
results_df.style.highlight_min(subset=['MAE', 'RMSE'], color='lightgreen').highlight_max(subset=['R²'], color='lightgreen')

## 6. Actual vs. Predicted — Visual Comparison

In [ ]:
# Pick the best model
best_name = results_df.drop('Baseline (7-day mean)', errors='ignore').index[0]
best_model, best_pred = trained_models[best_name]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter: actual vs predicted
ax1 = axes[0]
ax1.scatter(y_test, best_pred, alpha=0.3, s=10, color='teal')
max_val = max(y_test.max(), best_pred.max())
ax1.plot([0, max_val], [0, max_val], 'r--', alpha=0.7, label='Perfect prediction')
ax1.set_xlabel('Actual next_7_day_demand')
ax1.set_ylabel('Predicted next_7_day_demand')
ax1.set_title(f'{best_name} — Actual vs. Predicted', fontweight='bold')
ax1.legend()

# Residual histogram
ax2 = axes[1]
residuals = y_test.values - best_pred
ax2.hist(residuals, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax2.axvline(x=0, color='red', linestyle='--', alpha=0.7)
ax2.set_xlabel('Residual (Actual - Predicted)')
ax2.set_ylabel('Count')
ax2.set_title('Residual Distribution', fontweight='bold')
ax2.annotate(f'Mean: {np.mean(residuals):.2f}\nStd: {np.std(residuals):.2f}',
             xy=(0.72, 0.85), xycoords='axes fraction', fontsize=11,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig(os.path.join('..', 'reports', 'actual_vs_predicted.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Plot saved to reports/actual_vs_predicted.png')

## 7. Feature Importance

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importance = pd.Series(best_model.feature_importances_, index=features)
    top_15 = importance.sort_values(ascending=True).tail(15)

    fig, ax = plt.subplots(figsize=(10, 7))
    top_15.plot(kind='barh', ax=ax, color='teal', edgecolor='white')
    ax.set_title(f'{best_name} — Top 15 Feature Importances', fontsize=13, fontweight='bold')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.savefig(os.path.join('..', 'reports', 'feature_importance.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Plot saved to reports/feature_importance.png')
else:
    print(f'{best_name} does not have built-in feature_importances_ — use permutation importance instead.')

## 8. Save Best Model

In [ ]:
model_path = os.path.join('..', 'models', 'demand_model.pkl')
joblib.dump(best_model, model_path)
print(f'✅ Best model ({best_name}) saved to: {model_path}')
print(f'   MAE: {results[best_name]["MAE"]:.4f}')
print(f'   R²:  {results[best_name]["R²"]:.4f}')

## 9. Run Full Pipeline (Alternative)
Instead of this notebook, you can also run everything from the command line:
```bash
python src/demand_model.py
# or the full pipeline:
python src/run_pipeline.py
```
Both produce the same outputs: `demand_model.pkl`, `demand_model_card.md`, and `model_comparison.md`.